In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import BaggingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

In [10]:
# Preprocessing
df = pd.read_csv("energydata.csv")
print("Duplicated: ", df.duplicated().sum())
print()
print(df.isna().sum())
df.head()

Duplicated:  0

date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [11]:
# Preprocessing date feature
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df = df.drop(['date'], axis=1)
display(df[['year', 'month', 'day', 'hour', 'minute']])

,year,month,day,hour,minute
0,2016,1,11,17,0
1,2016,1,11,17,10
2,2016,1,11,17,20
3,2016,1,11,17,30
4,2016,1,11,17,40
...,...,...,...,...,...
19730,2016,5,27,17,20
19731,2016,5,27,17,30
19732,2016,5,27,17,40
19733,2016,5,27,17,50


In [12]:
# Splitting the dataset
X = df.drop(["Appliances"], axis=1)
y = df["Appliances"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
model_DTree = BaggingRegressor(DecisionTreeRegressor(), oob_score=True)
model_Line = BaggingRegressor(LinearRegression(), oob_score=True)
model_KNN = BaggingRegressor(KNeighborsRegressor(), oob_score=True)
model_MLP = BaggingRegressor(MLPRegressor(), oob_score=True)
model_SVR = BaggingRegressor(SVR(), oob_score=True)

In [ ]:
pipeline_dt = Pipeline([
    ("scaler", StandardScaler()),
    ("model", BaggingRegressor(estimator=DecisionTreeRegressor(), bootstrap=True, bootstrap_features=False, oob_score=True))
])

parameter_grid = {
    "model__n_estimators": [10, 25, 50, 75, 100],
    "model__max_samples": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
    "model__max_features": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
}

kf = KFold(n_splits=5, shuffle=True)

scores = ['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_root_mean_squared_error']

grid_search_dt = GridSearchCV(pipeline_dt, parameter_grid, cv=kf, scoring=scores, refit='neg_mean_squared_error', n_jobs=-2)

grid_search_dt.fit(X_train, y_train)

print("Best Parameters")
print(grid_search_dt.best_params_)
print()
print("Best Cross-Validation Accuracy")
print(grid_search_dt.best_score_)
print()
print("Best OOB Score")
print(grid_search_dt.best_estimator_.named_steps["model"].oob_score_)

results = pd.DataFrame(grid_search_dt.cv_results_)
results['mean_test_MAE'] = -results['mean_test_neg_mean_absolute_error']
results['mean_test_MSE'] = -results['mean_test_neg_mean_squared_error']
results['mean_test_RMSE'] = -results['mean_test_neg_root_mean_squared_error']
results.loc[grid_search_dt.best_index_][['params', 'mean_test_MAE', 'mean_test_MSE', 'mean_test_RMSE', 'mean_test_r2']]

Best Parameters
{'model__max_features': 0.5, 'model__max_samples': 1.0, 'model__n_estimators': 100}

Best Cross-Validation Accuracy
-5211.1113842571085

Best OOB Score
0.5342953714797538


params            {'model__max_features': 0.5, 'model__max_sampl...
mean_test_MAE                                             35.174689
mean_test_MSE                                           5211.111384
mean_test_RMSE                                            72.108926
mean_test_r2                                               0.509092
Name: 89, dtype: object

In [16]:
model_DTree = grid_search_dt.best_estimator_
y_pred = model_DTree.predict(X_test)
MAE_DT = mean_absolute_error(y_test, y_pred)
MSE_DT = mean_squared_error(y_test, y_pred)
RMSE_DT = root_mean_squared_error(y_test, y_pred)
print("MAE: ", MAE_DT)
print("MSE: ", MSE_DT)
print("RMSE: ", RMSE_DT)

MAE:  32.67719979733153
MSE:  4531.45923830434
RMSE:  67.3161142543473


In [18]:
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", BaggingRegressor(estimator=KNeighborsRegressor(), bootstrap=True, bootstrap_features=False, oob_score=True))
])

parameter_grid = {
    "model__n_estimators": [10, 25, 50, 75, 100],
    "model__max_samples": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
    "model__max_features": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
}

kf = KFold(n_splits=5, shuffle=True)

scores = ['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_root_mean_squared_error']

grid_search_knn = GridSearchCV(pipeline_knn, parameter_grid, cv=kf, scoring=scores, refit='neg_mean_squared_error', n_jobs=-2)

grid_search_knn.fit(X_train, y_train)

print("Best Parameters")
print(grid_search_knn.best_params_)
print()
print("Best Cross-Validation Accuracy")
print(grid_search_knn.best_score_)
print()
print("Best OOB Score")
print(grid_search_knn.best_estimator_.named_steps["model"].oob_score_)

results = pd.DataFrame(grid_search_knn.cv_results_)
results['mean_test_MAE'] = -results['mean_test_neg_mean_absolute_error']
results['mean_test_MSE'] = -results['mean_test_neg_mean_squared_error']
results['mean_test_RMSE'] = -results['mean_test_neg_root_mean_squared_error']
results.loc[grid_search_knn.best_index_][['params', 'mean_test_MAE', 'mean_test_MSE', 'mean_test_RMSE', 'mean_test_r2']]

Best Parameters
{'model__max_features': 0.25, 'model__max_samples': 1.0, 'model__n_estimators': 50}

Best Cross-Validation Accuracy
-6372.940739582593

Best OOB Score
0.4214527614296715


params            {'model__max_features': 0.25, 'model__max_samp...
mean_test_MAE                                             38.884174
mean_test_MSE                                            6372.94074
mean_test_RMSE                                            79.768775
mean_test_r2                                               0.398936
Name: 57, dtype: object

In [19]:
model_KNN = grid_search_knn.best_estimator_
y_pred = model_KNN.predict(X_test)
MAE_KNN = mean_absolute_error(y_test, y_pred)
MSE_KNN = mean_squared_error(y_test, y_pred)
RMSE_KNN = root_mean_squared_error(y_test, y_pred)
print("MAE: ", MAE_KNN)
print("MSE: ", MSE_KNN)
print("RMSE: ", RMSE_KNN)

MAE:  37.302131396723524
MSE:  5879.726000337781
RMSE:  76.6793714132933


In [20]:
pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", BaggingRegressor(estimator=LinearRegression(), bootstrap=True, bootstrap_features=False, oob_score=True))
])

parameter_grid = {
    "model__n_estimators": [10, 25, 50, 75, 100],
    "model__max_samples": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
    "model__max_features": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
}

kf = KFold(n_splits=5, shuffle=True)

scores = ['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error', 'neg_root_mean_squared_error']

grid_search_lr = GridSearchCV(pipeline_lr, parameter_grid, cv=kf, scoring=scores, refit='neg_mean_squared_error', n_jobs=-2)

grid_search_lr.fit(X_train, y_train)

print("Best Parameters")
print(grid_search_lr.best_params_)
print()
print("Best Cross-Validation Accuracy")
print(grid_search_lr.best_score_)
print()
print("Best OOB Score")
print(grid_search_lr.best_estimator_.named_steps["model"].oob_score_)

results = pd.DataFrame(grid_search_lr.cv_results_)
results['mean_test_MAE'] = -results['mean_test_neg_mean_absolute_error']
results['mean_test_MSE'] = -results['mean_test_neg_mean_squared_error']
results['mean_test_RMSE'] = -results['mean_test_neg_root_mean_squared_error']
results.loc[grid_search_lr.best_index_][['params', 'mean_test_MAE', 'mean_test_MSE', 'mean_test_RMSE', 'mean_test_r2']]

Best Parameters
{'model__max_features': 1.0, 'model__max_samples': 1.0, 'model__n_estimators': 10}

Best Cross-Validation Accuracy
-8861.430561917998

Best OOB Score
0.15436160906806196


c:\Users\VasuK\miniconda3\Lib\site-packages\sklearn\ensemble\_bagging.py:1464: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(


params            {'model__max_features': 1.0, 'model__max_sampl...
mean_test_MAE                                             53.031779
mean_test_MSE                                           8861.430562
mean_test_RMSE                                            93.989574
mean_test_r2                                                0.16289
Name: 175, dtype: object

In [21]:
model_LR = grid_search_lr.best_estimator_
y_pred = model_LR.predict(X_test)
MAE_LR = mean_absolute_error(y_test, y_pred)
MSE_LR = mean_squared_error(y_test, y_pred)
RMSE_LR = root_mean_squared_error(y_test, y_pred)
print("MAE: ", MAE_LR)
print("MSE: ", MSE_LR)
print("RMSE: ", RMSE_LR)

MAE:  52.69270385224877
MSE:  8556.607913719898
RMSE:  92.5019346485245


In [ ]:
results = pd.DataFrame(
    {"base_model": ["Decision Tree",
                    "Decision Tree",
                    "Decision Tree", 
                    "LinearRegression", 
                    "LinearRegression",
                    "LinearRegression",
                    "KNN Regression", 
                    "KNN Regression",
                    "KNN Regression",
                    "MLP Regression", 
                    "MLP Regression",
                    "MLP Regression",
                    "SVR", "SVR", "SVR"],
     "metric": ["MAE", "MSE", "RMSE", "MAE", "MSE", "RMSE", "MAE", "MSE", "RMSE", "MAE", "MSE", "RMSE", "MAE", "MSE", "RMSE",],
     "score": [MAE_DT, MSE_DT, RMSE_DT]}
    )